In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import KeyedVectors
from tqdm import tqdm
import re


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
class MyanmarTextPreprocessor():
    def __init__(self, dict_path: str, stop_path: str):
        self.dictionary = self.load_dictionary(dict_path)
        self.stopwords = self.load_stopwords(stop_path)

    def load_dictionary(self, dict_path):
        dictionary = set()
        with open(dict_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    dictionary.add(word)
        return dictionary

    def load_stopwords(self, stopword_path):
        stopwords = set()
        with open(stopword_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    stopwords.add(word)
        return stopwords

    def merge_with_dictionary(self, syllables):
        merged_tokens = []
        i = 0
        while i < len(syllables):
            matched = False
            for j in range(len(syllables), i, -1):
                combined = ''.join(syllables[i:j])
                if combined in self.dictionary:
                    merged_tokens.append(combined)
                    i = j
                    matched = True
                    break
            if not matched:
                merged_tokens.append(syllables[i])
                i += 1
        return merged_tokens

    def preprocessing(self, text: str):
        text = re.sub(r"(([A-Za-z0-9]+)|[က-အ|ဥ|ဦ](င်္|[က-အ][ှ]*[့း]*[်]|္[က-အ]|[ါ-ှႏꩻ][ꩻ]*){0,}|.)", r"\1 ", text)
        text = text.strip().split()
        merged_tokens = self.merge_with_dictionary(text)
        filtered_tokens = [token for token in merged_tokens if token not in self.stopwords]
        return ' '.join(filtered_tokens)

In [7]:
df = pd.read_csv("/content/drive/MyDrive/NLP Project/Headline Generator Dataset/headline_corpus.csv")

texts = df["text"].astype(str).tolist()
headlines = df["headline"].astype(str).tolist()


In [ ]:
import pickle
from multiprocessing import Pool
from pathlib import Path


DICT_PATH = "/content/drive/MyDrive/NLP Project/dict-words.txt"
STOPWORDS_PATH = "/content/drive/MyDrive/NLP Project/stopwords.txt"
CACHE_FILE = "preprocessed.pkl"

processor = MyanmarTextPreprocessor(DICT_PATH, STOPWORDS_PATH)
def preprocess_text(text, merge_dict=False):
    if merge_dict:
        # dictionary merge for headlines
        return processor.preprocessing(text).split()
    else:
        # syllable-level for long texts (fast)
        return re.sub(r"(([A-Za-z0-9]+)|[က-အ|ဥ|ဦ](င်္|[က-အ][ှ]*[့း]*[်]?|္[က-အ]|[ါ-ှ]*)*|.)", r"\1 ", text).strip().split()




Preprocessing texts and headlines in parallel...


In [ ]:
if Path(CACHE_FILE).exists():
    print("Loading cached preprocessed data...")
    with open(CACHE_FILE, "rb") as f:
        data = pickle.load(f)
        tokenized_texts = data["tokenized_texts"]
        tokenized_headlines = data["tokenized_headlines"]
else:
    print("Preprocessing texts and headlines in parallel...")
    with Pool(8) as p:
        tokenized_texts = p.starmap(preprocess_text, [(t, False) for t in texts])
        tokenized_headlines = p.starmap(preprocess_text, [(h, True) for h in headlines])

    # Save cache
    with open(CACHE_FILE, "wb") as f:
        pickle.dump({
            "tokenized_texts": tokenized_texts,
            "tokenized_headlines": tokenized_headlines
        }, f)

In [ ]:
print("Preprocessing complete.")
print("Sample tokens (text):", tokenized_texts[0])
print("Sample tokens (headline):", tokenized_headlines[0])

In [ ]:
print("Loading fastText vectors...")
ft = KeyedVectors.load_word2vec_format("/content/drive/MyDrive/NLP Project/Headline Generator Dataset/cc.my.300.vec")
embedding_dim = 300


In [ ]:
from collections import Counter

# --- Load fastText embeddings ---
print("Loading fastText vectors...")
ft = KeyedVectors.load_word2vec_format("/content/drive/MyDrive/cc.my.300.vec")
embedding_dim = 300

# --- Build vocabulary from preprocessed tokens ---
counter = Counter()

# texts and headlines are already preprocessed (space-separated)
for t in texts + headlines:
    counter.update(tokenize(t))

# Keep only words that appear at least twice
vocab = ["<pad>", "<unk>", "<sos>", "<eos>"] + [w for w, c in counter.items() if c >= 2]

# --- Mappings ---
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print("Vocab size:", vocab_size)


In [ ]:
embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embedding_dim))

for word, idx in word2idx.items():
    if word in ft:
        embedding_matrix[idx] = ft[word]

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float)


In [ ]:
import matplotlib.pyplot as plt

text_lengths = [len(tokenize(t)) for t in texts]
headline_lengths = [len(tokenize(h)) for h in headlines]
plt.hist(text_lengths, bins=50)
plt.title("Text token lengths")
plt.show()

plt.hist(headline_lengths, bins=20)
plt.title("Headline token lengths")
plt.show()


In [ ]:
EMBEDDING_DIM = 300
HIDDEN_DIM = 256
NUM_LAYERS = 2
BATCH_SIZE = 32
MAX_TEXT_LEN = 200
MAX_HEAD_LEN = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def encode_sentence(sentence, max_len, add_sos_eos=False):
    tokens = tokenize(sentence)
    ids = []

    if add_sos_eos:
        ids.append(word2idx["<sos>"])

    for tok in tokens[:max_len]:
        ids.append(word2idx.get(tok, word2idx["<unk>"]))

    if add_sos_eos:
        ids.append(word2idx["<eos>"])

    if len(ids) < max_len:
        ids += [word2idx["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids


In [ ]:
class HeadlineDataset(Dataset):
    def __init__(self, texts, headlines):
        self.texts = texts
        self.headlines = headlines

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        src = torch.tensor(encode_sentence(self.texts[idx], MAX_TEXT_LEN), dtype=torch.long)
        trg = torch.tensor(encode_sentence(self.headlines[idx], MAX_HEAD_LEN, add_sos_eos=True), dtype=torch.long)
        decoder_input = trg[:-1]
        decoder_target = trg[1:]
        return src, decoder_input, decoder_target

dataset = HeadlineDataset(texts, headlines)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx["<pad>"])
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
            self.embedding.weight.requires_grad = True  # fine-tune embeddings

        self.encoder = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=False)
        self.decoder = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src, trg_input):
        # src: (batch, MAX_TEXT_LEN)
        # trg_input: (batch, MAX_HEAD_LEN-1)

        # Encoder
        embedded_src = self.embedding(src)  # (batch, seq_len, embed_dim)
        _, (hidden, cell) = self.encoder(embedded_src)

        # Decoder
        embedded_trg = self.embedding(trg_input)  # (batch, seq_len, embed_dim)
        outputs, _ = self.decoder(embedded_trg, (hidden, cell))  # initialize decoder with encoder hidden state

        # Final output
        logits = self.fc(outputs)  # (batch, seq_len, vocab_size)
        return logits

In [ ]:
model = Seq2SeqLSTM(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, embedding_matrix=embedding_matrix)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<pad>"])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(1):
    model.train()
    total_loss = 0
    for src, dec_input, dec_target in loader:
        src = src.to(DEVICE)
        dec_input = dec_input.to(DEVICE)
        dec_target = dec_target.to(DEVICE)

        optimizer.zero_grad()
        output = model(src, dec_input)  # (batch, seq_len, vocab_size)

        # Flatten for loss: (batch*seq_len, vocab_size)
        loss = criterion(output.view(-1, vocab_size), dec_target.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

In [ ]:
def generate_headline(model, text, max_len=MAX_HEAD_LEN, temperature=1.0):
    """
    Generate a headline for a given article text.

    Args:
        model: trained Seq2Seq LSTM
        text: input article (string)
        max_len: maximum headline length
        temperature: controls randomness (1.0=normal, <1=less random, >1=more random)

    Returns:
        generated headline (string)
    """
    model.eval()
    with torch.no_grad():
        # --- Preprocess input text ---
        src_ids = torch.tensor(encode_sentence(text, MAX_TEXT_LEN), dtype=torch.long).unsqueeze(0).to(DEVICE)  # (1, seq_len)

        # --- Encoder ---
        embedded_src = model.embedding(src_ids)
        _, (hidden, cell) = model.encoder(embedded_src)

        # --- Decoder: start with <sos> ---
        input_token = torch.tensor([[word2idx["<sos>"]]], dtype=torch.long).to(DEVICE)
        generated_ids = []

        for _ in range(max_len):
            embedded = model.embedding(input_token)
            output, (hidden, cell) = model.decoder(embedded, (hidden, cell))
            logits = model.fc(output[:, -1, :])  # last timestep
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)

            # Sample next word
            next_id = torch.argmax(probs, dim=-1).item()
            if next_id == word2idx["<eos>"]:
                break
            generated_ids.append(next_id)

            # Prepare next input
            input_token = torch.tensor([[next_id]], dtype=torch.long).to(DEVICE)

        # Convert IDs back to words
        generated_words = [idx2word[i] for i in generated_ids]
        return " ".join(generated_words)


In [ ]:
import torch
import pickle

# ----------------------------
# Save model weights
# ----------------------------
torch.save(model.state_dict(), "seq2seq_model.pth")

# ----------------------------
# Save vocab mappings
# ----------------------------
with open("vocab.pkl", "wb") as f:
    pickle.dump({
        "word2idx": word2idx,
        "idx2word": idx2word
    }, f)

# ----------------------------
# Save processor paths (dictionary + stopwords)
# ----------------------------
processor_info = {
    "dict_path": "dict.txt",
    "stopwords_path": "stopwords.txt"
}

with open("processor.pkl", "wb") as f:
    pickle.dump(processor_info, f)

# ----------------------------
# Optional: save embedding matrix
# ----------------------------
torch.save(embedding_matrix, "embedding_matrix.pt")
